# SC-13-Fuzz-Invariants - Fuzz Testing

[<< Precedent : Foundry Testing](SC-12-Foundry-Testing.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Formal Verification >>](SC-14-Formal-Verification.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre le **fuzz testing** et ses avantages
2. Ecrire des tests avec **parametres aleatoires**
3. Tester des **invariants** de contrats
4. Utiliser `vm.assume` pour filtrer les entrees

### Prerequis

- [SC-12-Foundry-Testing](SC-12-Foundry-Testing.ipynb) complete
- Foundry installe et projet initialise
- Notions de tests unitaires en Solidity

### Duree estimee : 40 minutes

---

## 1. Introduction au Fuzz Testing

Le fuzz testing genere automatiquement des entrees aleatoires pour trouver des bugs.

In [1]:
# Concept de fuzz testing
print("""
FUZZ TESTING - Concepts cles

1. GENERATION ALEATOIRE
   - Foundry genere des valeurs aleatoires pour chaque parametre
   - Types supportes: uint, int, address, bytes, string, arrays

2. CAS D'USAGE
   - Trouver des edge cases non couverts par les tests unitaires
   - Tester des bornes (overflow, underflow)
   - Verifier des invariants mathematiques

3. CONFIGURATION
   - runs: nombre d'executions (defaut: 256)
   - depth: profondeur des appels internes
   - seed: pour reproduire un bug specifique

4. COMMANDE
   forge test --fuzz-runs 1000 --match-path test/Fuzz
""")


FUZZ TESTING - Concepts cles

1. GENERATION ALEATOIRE
   - Foundry genere des valeurs aleatoires pour chaque parametre
   - Types supportes: uint, int, address, bytes, string, arrays

2. CAS D'USAGE
   - Trouver des edge cases non couverts par les tests unitaires
   - Tester des bornes (overflow, underflow)
   - Verifier des invariants mathematiques

3. CONFIGURATION
   - runs: nombre d'executions (defaut: 256)
   - depth: profondeur des appels internes
   - seed: pour reproduire un bug specifique

4. COMMANDE
   forge test --fuzz-runs 1000 --match-path test/Fuzz



---

## 2. Fuzz Test Simple

In [2]:
# Fuzz test basique
FUZZ_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SafeMath {
    // Fonction a tester
    function safeAdd(uint256 a, uint256 b) public pure returns (uint256) {
        uint256 c = a + b;
        require(c >= a, "Overflow");
        return c;
    }
    
    function safeMul(uint256 a, uint256 b) public pure returns (uint256) {
        if (a == 0) return 0;
        uint256 c = a * b;
        require(c / a == b, "Overflow");
        return c;
    }
}
'''

FUZZ_TEST_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SafeMath.sol";

contract SafeMathFuzzTest is Test {
    SafeMath sm;

    function setUp() public {
        sm = new SafeMath();
    }

    // Fuzz test: a et b sont generes aleatoirement
    function testFuzz_SafeAdd(uint256 a, uint256 b) public pure {
        // Si pas d'overflow, safeAdd doit reussir
        if (a + b >= a) {
            assertEq(sm.safeAdd(a, b), a + b);
        }
    }

    // Fuzz test avec contrainte
    function testFuzz_SafeMul(uint256 a, uint256 b) public pure {
        vm.assume(a > 0);  // Eviter division par zero
        vm.assume(b > 0);  // Cas non-trivial
        
        uint256 result = sm.safeMul(a, b);
        assertEq(result, a * b);
    }

    // Test avec bounds explicites
    function testFuzz_SafeAdd_Bounds() public pure {
        // Tester avec max uint256
        vm.assume(uint128(a) + uint128(b) == uint128(a) + uint128(b));
        assertEq(sm.safeAdd(a, b), a + b);
    }
}
'''

print("Contrat SafeMath:")
print(FUZZ_BASIC)
print("\n" + "="*50 + "\n")
print("Test Fuzz:")
print(FUZZ_TEST_BASIC)

Contrat SafeMath:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SafeMath {
    // Fonction a tester
    function safeAdd(uint256 a, uint256 b) public pure returns (uint256) {
        uint256 c = a + b;
        require(c >= a, "Overflow");
        return c;
    }

    function safeMul(uint256 a, uint256 b) public pure returns (uint256) {
        if (a == 0) return 0;
        uint256 c = a * b;
        require(c / a == b, "Overflow");
        return c;
    }
}



Test Fuzz:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SafeMath.sol";

contract SafeMathFuzzTest is Test {
    SafeMath sm;

    function setUp() public {
        sm = new SafeMath();
    }

    // Fuzz test: a et b sont generes aleatoirement
    function testFuzz_SafeAdd(uint256 a, uint256 b) public pure {
        // Si pas d'overflow, safeAdd doit reussir
        if (a + b >= a) {
            assertEq(sm.safeAdd(a, b), a + b);
        }
    }

   

["### Interpretation : Concepts du Fuzz Testing\n", "\n", "Cette synthese presente les principes fondamentaux du fuzz testing avec Foundry.\n", "\n", "| Concept | Description | Impact sur les tests |\n", "|---------|-------------|---------------------|\n", "| **Generation aleatoire** | Foundry genere des valeurs pour tous les parametres `uint`/`int`/`address`/`bytes`/`arrays` | Couvre des cas auxquels le developpeur n'a pas pense |\n", "| **Edge cases** | Bornes extremes (0, max uint256, overflow, underflow) | Trouve des bugs de debordement arithmetique |\n", "| **Invariants mathematiques** | Proprietes qui doivent toujours etre vraies (ex: somme des balances = total supply) | Valide la coherence de l'etat du contrat |\n", "| **Configuration `runs`** | Nombre d'iterations par test (defaut 256) | Plus de runs = plus de couverture, mais plus lent |\n", "\n", "**Points cles** :\n", "1. Le fuzz testing est complementaire aux tests unitaires : il explore l'espace des inputs de maniere systematique et aleatoire\n", "2. `depth` dans la configuration correspond a la profondeur des appels de fonctions imbriques (pour les invariants)\n", "3. Le parametre `seed` permet la reproductibilite : meme bug, meme seed = meme echec\n", "4. La commande `forge test --fuzz-runs 1000` augmente la couverture pour des tests plus exhaustifs\n", "\n", "**Cas d'usage typiques** :\n", "- Tester des fonctions avec des parametres utilisateurs non controlees\n", "- Valider des contraintes d'access control (roles, permissions)\n", "- Verifier la coherence d'invariants comptables (balances, supply, ratios)\n", "\n", "> **Note technique** : Le fuzz testing avec Foundry est \"stateless\" par defaut (chaque test repart de zero). Pour du \"stateful fuzzing\" (plusieurs transactions chainees), il faut utiliser la section `[invariant]` de foundry.toml.\n"]

---

## 3. Invariant Testing

Les invariants testent que des proprietes restent toujours vraies.

In [3]:
# Invariant testing pour un token ERC-20
INVARIANT_TEST = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SimpleToken.sol";

contract TokenInvariantTest is Test {
    SimpleToken token;
    address[] users;

    function setUp() public {
        token = new SimpleToken(1_000_000 * 10**18);
        
        // Creer des utilisateurs
        for (uint i = 0; i < 5; i++) {
            users.push(address(uint160(i + 1)));
        }
    }

    // Invariant: le total supply ne change jamais
    function invariant_TotalSupply() public view {
        uint256 initialSupply = token.totalSupply();
        assertEq(token.totalSupply(), initialSupply);
    }

    // Invariant: sum of balances == total supply
    function invariant_BalanceSum() public view {
        uint256 sum = 0;
        for (uint i = 0; i < users.length; i++) {
            sum += token.balanceOf(users[i]);
        }
        // Note: peut echouer si mint/burn existe
    }
}
'''

print("Test d'Invariants:")
print(INVARIANT_TEST)

Test d'Invariants:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SimpleToken.sol";

contract TokenInvariantTest is Test {
    SimpleToken token;
    address[] users;

    function setUp() public {
        token = new SimpleToken(1_000_000 * 10**18);

        // Creer des utilisateurs
        for (uint i = 0; i < 5; i++) {
            users.push(address(uint160(i + 1)));
        }
    }

    // Invariant: le total supply ne change jamais
    function invariant_TotalSupply() public view {
        uint256 initialSupply = token.totalSupply();
        assertEq(token.totalSupply(), initialSupply);
    }

    // Invariant: sum of balances == total supply
    function invariant_BalanceSum() public view {
        uint256 sum = 0;
        for (uint i = 0; i < users.length; i++) {
            sum += token.balanceOf(users[i]);
        }
        // Note: peut echouer si mint/burn existe
    }
}



["### Interpretation : Fuzz Tests pour Operations Arithmetiques Securisees\n", "\n", "Les tests presentent des strategies de fuzzing pour valider les protections contre overflow/underflow.\n", "\n", "| Test | Strategie | Validation |\n", "|------|-----------|------------|\n", "| `testFuzz_SafeAdd` | Cas general | Verifie que le resultat correspond a l'addition native si pas d'overflow |\n", "| `testFuzz_SafeMul` | Filtrage inputs | `vm.assume` evite les cas triviaux (a=0 ou b=0) |\n", "| `testFuzz_SafeAdd_Bounds` | Bornes explicites | Teste uniquement le cas uint128 (pas d'overflow possible) |\n", "\n", "**Points cles** :\n", "1. `testFuzz_SafeAdd` utilise une condition `if (a + b >= a)` pour detecter l'overflow AVANT d'appeler la fonction\n", "2. `testFuzz_SafeMul` combine `vm.assume(a > 0)` et `vm.assume(b > 0)` pour eviter les cas edge (division par zero dans la verification)\n", "3. Le pattern `assertEq(result, a * b)` verifie que le resultat correspond a l'operation native (qui peut overflow en Solidity 0.8+)\n", "4. `testFuzz_SafeAdd_Bounds` montre une approche conservative : limiter les inputs a une plage safe (uint128)\n", "\n", "**Difference fondamentale** : Solidity 0.8+ a des checks d'overflow automatiques (`0x...` revert), mais les tests fuzz valident que la logique de protection (`require`) fonctionne correctement.\n", "\n", "> **Note technique** : Le fuzzing trouve des bugs que les tests unitaires manquent. Par exemple, un test unitaire pourrait tester `1 + 1 = 2`, mais le fuzz testera `2^256 - 1 + 1` et detectera l'overflow.\n"]

---

## 4. vm.assume vs require

| Instruction | Usage | Effet |
|-------------|------|------|
| `vm.assume(cond)` | Dans les fuzz tests | Rejette l'input et genere une nouvelle |
| `require(cond, msg)` | Dans le contrat | Revert la transaction (echec du test) |

In [4]:
# Exemple avec vm.assume
ASSUME_EXAMPLE = '''
contract AssumeExample is Test {
    // Fuzz test avec filtrage d'inputs
    function testFuzz_Filtered(
        uint256 a,
        uint256 b,
        address addr
    ) public pure {
        // Filtrer les cas non desires
        vm.assume(a > 0);           // a non nul
        vm.assume(b > 0);           // b non nul
        vm.assume(a < type(uint128).max);  // a dans les limites
        vm.assume(addr != address(0));  // adresse non nulle
        vm.assume(addr.code.length == 0);  // pas un contrat
        
        // Maintenant tester avec des entrees valides
        uint256 result = a + b;
        assertGt(result, a);
        assertGt(result, b);
    }

    // Pattern commun de filtrage
    function testFuzz_AddressFilter(address addr) public pure {
        vm.assume(addr != address(0));
        vm.assume(addr != 0x0000000000000000000000000000000000000001);  // Ecrou
        // ... test logic
    }

    function testFuzz_UintRange(uint256 x) public pure {
        vm.assume(x >= 100 && x <= 1000);
        // x est maintenant dans [100, 1000]
    }
}
'''

print("Patterns avec vm.assume:")
print(ASSUME_EXAMPLE)

Patterns avec vm.assume:

contract AssumeExample is Test {
    // Fuzz test avec filtrage d'inputs
    function testFuzz_Filtered(
        uint256 a,
        uint256 b,
        address addr
    ) public pure {
        // Filtrer les cas non desires
        vm.assume(a > 0);           // a non nul
        vm.assume(b > 0);           // b non nul
        vm.assume(a < type(uint128).max);  // a dans les limites
        vm.assume(addr != address(0));  // adresse non nulle
        vm.assume(addr.code.length == 0);  // pas un contrat

        // Maintenant tester avec des entrees valides
        uint256 result = a + b;
        assertGt(result, a);
        assertGt(result, b);
    }

    // Pattern commun de filtrage
    function testFuzz_AddressFilter(address addr) public pure {
        vm.assume(addr != address(0));
        vm.assume(addr != 0x0000000000000000000000000000000000000001);  // Ecrou
        // ... test logic
    }

    function testFuzz_UintRange(uint256 x) public pure {
      

["### Interpretation : Tests d'Invariants pour Token ERC-20\n", "\n", "Ce code presente des tests de proprietes qui doivent toujours etre vraies pour un token ERC-20.\n", "\n", "| Invariant | Propriete testee | Implementation |\n", "|-----------|-----------------|----------------|\n", "| `invariant_TotalSupply` | Le supply total est constant | Verifie que `totalSupply()` ne change jamais |\n", "| `invariant_BalanceSum` | Somme des balances = supply | Additionne tous les balances et compare au supply |\n", "\n", "**Points cles** :\n", "1. Les fonctions `invariant_*` sont appelees automatiquement par Foundry entre chaque operation aleatoire\n", "2. `invariant_TotalSupply` est valable uniquement s'il n'y a pas de fonctions `mint`/`burn`\n", "3. `invariant_BalanceSum` est l'invariant fondamental d'ERC-20 : conservation de la masse monetaire\n", "4. La boucle `for` dans `invariant_BalanceSum` ne couvre que 5 utilisateurs predefinis (pas exhaustif)\n", "\n", "**Limitation** : Le test note que `invariant_BalanceSum` peut echouer si des fonctions `mint`/`burn` existent, car elles brisent l'invariant de conservation (ce qui est normal pour ces operations).\n", "\n", "> **Note technique** : Foundry execute les invariants en appelant des fonctions du contrat de maniere aleatoire (fuzzing oriente etat), puis verifie que les invariants restent valides. La `depth` dans foundry.toml controle combien d'appels sont chaines.\n"]

---

## 5. Configuration Fuzz

Dans `foundry.toml`:

In [5]:
# Configuration foundry.toml
FOUNDRY_CONFIG = '''
[fuzz]
runs = 1000          # Nombre d'executions par test
max_test_rejects = 1000  # Max rejets avant abandon
seed = '0x1234'    # Seed pour reproductibilite
dictionary_weight = 40  # Poids du dictionnaire

[invariant]
runs = 256           # Runs par invariant
depth = 15           # Profondeur des appels
fail_on_revert = true  # Echouer sur revert
'''

# Commandes
COMMANDS = '''
# Executer les fuzz tests
forge test --fuzz-runs 5000

# Executer uniquement les tests fuzz
forge test --match-path "test/*Fuzz*"

# Avec un seed specifique
forge test --fuzz-seed 0x3e8e

# Debug mode
forge test -vvvv --fuzz-runs 10
'''

print("Configuration foundry.toml:")
print(FOUNDRY_CONFIG)
print("\n" + "="*50 + "\n")
print("Commandes:")
print(COMMANDS)

Configuration foundry.toml:

[fuzz]
runs = 1000          # Nombre d'executions par test
max_test_rejects = 1000  # Max rejets avant abandon
seed = '0x1234'    # Seed pour reproductibilite
dictionary_weight = 40  # Poids du dictionnaire

[invariant]
runs = 256           # Runs par invariant
depth = 15           # Profondeur des appels
fail_on_revert = true  # Echouer sur revert



Commandes:

# Executer les fuzz tests
forge test --fuzz-runs 5000

# Executer uniquement les tests fuzz
forge test --match-path "test/*Fuzz*"

# Avec un seed specifique
forge test --fuzz-seed 0x3e8e

# Debug mode
forge test -vvvv --fuzz-runs 10



["### Interpretation : Patterns vm.assume\n", "\n", "Les exemples montrent comment filtrer efficacement les inputs aleatoires pour se concentrer sur les cas interessants.\n", "\n", "| Pattern | Usage | Exemple |\n", "|---------|-------|----------|\n", "| `vm.assume(x > 0)` | Exclure zero | Eviter division par zero |\n", "| `vm.assume(x < bound)` | Borner les valeurs | Garder dans uint128 pour eviter overflow |\n", "| `vm.assume(addr.code.length == 0)` | EOA only | Exclure les contrats intelligents |\n", "| `vm.assume(x >= 100 && x <= 1000)` | Range specifique | Tester uniquement une plage pertinente |\n", "\n", "**Points cles** :\n", "1. `vm.assume` ne fait pas echouer le test : elle rejette l'input et en genere un nouveau\n", "2. Trop de `vm.assume` ralentit le fuzz (beaucoup d'inputs rejetes)\n", "3. L'adresse `0x0000000000000000000000000000000000000001` est un \"ecrou\" (precompile) souvent exclue\n", "4. `addr.code.length` distingue EOA (0) de contrats (>0) : utile pour les fonctions `payable`\n", "\n", "> **Note technique** : La difference fondamentale : `vm.assume` filtre les inputs AVANT execution (fuzzing), `require` fait echouer le test PENDANT execution (contract logic).\n"]

---

## 6. Exercices

In [6]:
# Exercice 1 : Fuzzer pour une pile LIFO
# TODO etudiant : ecrire des fuzz tests pour verifier les proprietes d'une pile LIFO
# Indice : quels invariants une pile LIFO doit-elle respecter ?
# Etape 1 : implementer testFuzz_PushPop - verifier que push puis pop laisse la taille inchangee
# Etape 2 : implementer testFuzz_LIFO - verifier que le dernier element empile est le premier depile

EXERCISE_STACK = '''
// Contrat a tester
contract Stack {
    uint256[] private items;
    
    function push(uint256 item) public {
        items.push(item);
    }
    
    function pop() public returns (uint256) {
        require(items.length > 0, "Empty stack");
        return items.pop();
    }
    
    function peek() public view returns (uint256) {
        require(items.length > 0, "Empty stack");
        return items[items.length - 1];
    }
    
    function size() public view returns (uint256) {
        return items.length;
    }
}

// Fuzz test
contract StackFuzzTest is Test {
    Stack stack;
    
    function setUp() public {
        stack = new Stack();
    }
    
    // Invariant: apres push puis pop, size est inchange
    function testFuzz_PushPop(uint256 value) public {
        // TODO: etudiant - verifier que push(value) incremente size de 1
        // TODO: etudiant - verifier que pop() retourne value
        // TODO: etudiant - verifier que size revient a la valeur initiale
    }
    
    // Invariant: LIFO property
    function testFuzz_LIFO(uint256[] memory values) public {
        // TODO: etudiant - filtrer les valeurs avec vm.assume (length > 0, <= 100)
        // TODO: etudiant - push tous les elements
        // TODO: etudiant - pop et verifier LIFO (dernier empile = premier depile)
        // Indice : parcourir le tableau a l'envers pour verifier
    }
}
'''
print("Exercice Stack Fuzz Test - a completer")
print("Implementez les tests fuzz marques TODO dans EXERCISE_STACK")


Exercice Stack Fuzz Test - a completer
Implementez les tests fuzz marques TODO dans EXERCISE_STACK


["### Indice : Exercice Stack Fuzz Test\n", "\n", "Pour implementer les tests fuzz de la pile LIFO, rappelez-vous les proprietes fondamentales d'une pile :\n", "\n", "**Invariant 1 - Push/Pop** :\n", "- La taille de la pile augmente de 1 apres chaque `push`\n", "- La taille diminue de 1 apres chaque `pop`\n", "- Apres `push(x)` puis `pop()`, la taille revient a sa valeur initiale\n", "\n", "**Invariant 2 - LIFO (Last In, First Out)** :\n", "- L'element depile par `pop()` est toujours le dernier empile\n", "- Si on empile `[a, b, c]`, alors `pop()` retourne `c`, puis `b`, puis `a`\n", "- Parcourir le tableau a l'envers (de la fin au debut) donne l'ordre de depilement\n", "\n", "**Indice implementation** :\n", "- Utiliser `vm.assume(values.length > 0 && values.length <= 100)` pour borner la taille\n", "- Utiliser une boucle `for (uint i = 0; i < values.length; i++)` pour les pushes\n", "- Utiliser une boucle `for (uint i = values.length; i > 0; i--)` pour les pops et verifier LIFO\n", "- Verifier avec `assertEq(pop(), values[i - 1])` que l'ordre est respecte\n"]

---

## 7. Resume

| Concept | Description |
|---------|-------------|
| Fuzz test | Test avec parametres aleatoires |
| `vm.assume` | Filtrer les entrees non desirees |
| Invariant | Propriete toujours vraie |
| `runs` | Nombre d'executions par test |

### Bonnes pratiques
1. Toujours limiter les ranges avec `vm.assume`
2. Tester les edge cases separement
3. Utiliser des seeds pour reproduire les bugs
4. Combiner fuzz tests et tests unitaires

---

**Notebook suivant** : [SC-14-Formal-Verification](SC-14-Formal-Verification.ipynb)

---

[<< Precedent : Foundry Testing](SC-12-Foundry-Testing.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Formal Verification >>](SC-14-Formal-Verification.ipynb)

["### Interpretation : Configuration Fuzz Foundry\n", "\n", "La configuration trouvee definit comment Foundry execute les tests de fuzzing et d'invariants.\n", "\n", "| Parametre | Valeur | Signification |\n", "|-----------|--------|---------------|\n", "| `runs = 1000` | Nombre d'iterations | Chaque test fuzz est execute 1000 fois avec des inputs differents |\n", "| `max_test_rejects = 1000` | Limite de rejets | Si `vm.assume` rejette plus de 1000 inputs, le test est abandonne |\n", "| `seed = '0x1234'` | Graine aleatoire | Permet de reproduire exactement le meme fuzz test (reproductibilite des bugs) |\n", "| `depth = 15` | Profondeur invariants | Pour les invariants, nombre d'appels de fonctions aleatoires |\n", "| `fail_on_revert = true` | Comportement revert | Un invariant echoue si une fonction revert |\n", "\n", "**Points cles** :\n", "1. `runs = 1000` est un bon compromis entre couverture et temps d'execution (defaut 256)\n", "2. `seed` est critique pour le debugging : permet de rejouer exactement le meme cas qui a revele un bug\n", "3. `max_test_rejects` evite les boucles infinies quand les contraintes `vm.assume` sont trop restrictives\n", "4. La section `[invariant]` est specifique aux tests de proprietes (differents des tests fuzz simples)\n", "\n", "> **Note technique** : En production, on utilise souvent `runs = 10000` ou plus pour les contrats critiques. Le `--fuzz-runs` en ligne de commande surcharge la config.\n"]